# DialogSum: GPU pipeline (Colab)

Ноутбук только вызывает скрипты из репозитория: подготовка данных → бейзлайн → zero-shot через vLLM → LoRA (unsloth + trl) → инференс LoRA через vLLM → метрики → LLM-as-judge.

**Перед запуском:** `Runtime → Change runtime type → T4 GPU`. В Secrets (ключик слева) по желанию добавить `WANDB_API_KEY` и `GROQ_API_KEY`. Потом `Runtime → Run all`. Все артефакты сохраняются на Google Drive в `MyDrive/text-generation/outputs`.

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

def run(command):
    print(f"Running: {command}", flush=True)
    subprocess.run(shlex.split(command), check=True)

run('nvidia-smi')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
os.chdir('/content')
repo = Path('/content/text-generation')
branch = 'feature/lora-summarization'
if not repo.exists():
    run(f'git clone -b {branch} https://github.com/semenovfilipp/text-generation.git')
os.chdir(repo)
run('git fetch origin')
run(f'git checkout {branch}')
run(f'git pull --ff-only origin {branch}')
run('git rev-parse HEAD')
drive_outputs = Path('/content/drive/MyDrive/text-generation/outputs')
drive_outputs.mkdir(parents=True, exist_ok=True)
outputs = repo / 'outputs'
if outputs.is_symlink():
    assert outputs.resolve() == drive_outputs.resolve(), 'outputs points to another directory'
elif outputs.exists():
    raise RuntimeError('outputs already exists as a directory; move it before rerunning')
else:
    outputs.symlink_to(drive_outputs, target_is_directory=True)


## Environment

Each step runs in a separate process and checks its exit code. Stop at any error before continuing. Installed versions are saved to `outputs/environment.txt`.


In [ ]:
run('python -m pip install -q unsloth vllm')
run('python -m pip install -q -r requirements.txt wandb')
run('python -m pip install -q -e .')
run('python -c "import unsloth; import torch, vllm, trl, peft; print(torch.__version__, vllm.__version__, trl.__version__, peft.__version__)"')
with (outputs / 'environment.txt').open('w') as f:
    subprocess.run(['python', '-m', 'pip', 'freeze'], check=True, stdout=f)


In [ ]:
import os
from google.colab import userdata

for name in ['WANDB_API_KEY', 'GROQ_API_KEY', 'KAGGLE_USERNAME', 'KAGGLE_KEY', 'HF_TOKEN']:
    try:
        os.environ[name] = userdata.get(name)
        print(name, 'loaded')
    except Exception:
        print(name, 'not set')

## Данные и бейзлайн без ML

In [ ]:
run('python scripts/prepare_data.py --config configs/data.yaml')
run('python scripts/baseline_lead.py --config configs/data.yaml')


## Zero-shot Qwen2.5-0.5B-Instruct через vLLM

In [ ]:
run('python scripts/infer.py --config configs/infer_zeroshot.yaml --debug')
run('python scripts/infer.py --config configs/infer_zeroshot.yaml')


## LoRA-дообучение (unsloth + trl SFTTrainer)

Если unsloth не встанет, запасной вариант: `!python scripts/train_lora.py --config configs/train_lora.yaml --backend peft` (чистые trl + peft).

In [ ]:
run('python scripts/train_lora.py --config configs/train_lora.yaml --debug')
run('python scripts/train_lora.py --config configs/train_lora.yaml')


In [ ]:
from IPython.display import Image, display
import json

display(Image('outputs/models/qwen2.5-0.5b-lora/loss.png'))
print(json.load(open('outputs/models/qwen2.5-0.5b-lora/train_summary.json')))

## Инференс LoRA-модели через vLLM

In [ ]:
run('python scripts/infer.py --config configs/infer_lora.yaml')


## Метрики

In [ ]:
run('python scripts/evaluate.py --output outputs/metrics.csv')


In [ ]:
import pandas as pd
from dialogsum.utils import read_jsonl, write_json

test = {r['id']: r for r in read_jsonl('data/processed/test.jsonl')}
preds = {n: {p['id']: p['prediction'] for p in read_jsonl(f'outputs/predictions/{n}.jsonl')} for n in ['lead_first_last', 'qwen2.5-0.5b_zeroshot', 'qwen2.5-0.5b_lora']}
examples = []
for i in ['test_0', 'test_7', 'test_42', 'test_100', 'test_250']:
    item = {**test[i], 'predictions': {n: p[i] for n, p in preds.items()}, 'review': 'TODO: factual errors, coverage, brevity'}
    examples.append(item)
    print(item)
write_json(examples, 'outputs/manual_review.json')


## LLM-as-judge (опционально, если задан `GROQ_API_KEY`)

Если ключа в Secrets нет, пропусти ячейку: судью можно запустить локально по предсказаниям с Drive.

In [ ]:
import yaml
judge_cfg = yaml.safe_load(Path('configs/judge.yaml').read_text())
judge_cfg['output_dir'] = 'outputs/judge'
Path('/content/judge-colab.yaml').write_text(yaml.safe_dump(judge_cfg))
if os.environ.get('GROQ_API_KEY'):
    run('python scripts/judge.py --config /content/judge-colab.yaml --debug')
    run('python scripts/judge.py --config /content/judge-colab.yaml')
else:
    print('GROQ_API_KEY not set: judge is still required to complete the assignment')


## Сохранить результаты

Скачай `MyDrive/text-generation/colab_results.zip`: там предсказания, метрики, логи и график обучения, без весов.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
archive = drive_outputs.parent / 'colab_results.zip'
files = [outputs / 'metrics.csv', outputs / 'environment.txt', outputs / 'manual_review.json']
files += list((outputs / 'predictions').glob('*'))
files += list((outputs / 'judge').glob('*'))
model_dir = outputs / 'models/qwen2.5-0.5b-lora'
files += [model_dir / name for name in ['log_history.json', 'loss.png', 'train_summary.json']]
with ZipFile(archive, 'w', ZIP_DEFLATED) as z:
    for file in files:
        if file.is_file():
            z.write(file, file.relative_to(outputs))
print(f'Results saved: {archive}')
